## 📊 Exploratory Data Analysis (EDA) & Cleaning

### Importing Libraries and Loading the Dataset

In [85]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Visualisation Setup

In [86]:
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

In [87]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx'
df_raw = pd.read_excel(url, engine='openpyxl')

In [88]:
print(f'Rows:    {df_raw.shape[0]:>10,}')
print(f'Columns: {df_raw.shape[1]:>10}')
df_raw.head(5)

Rows:       541,909
Columns:          8


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.00,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.00,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom


In [89]:
info_df = pd.DataFrame({
    'dtype': df_raw.dtypes,
    'non_null': df_raw.count(),
    'null_count': df_raw.isnull().sum(),
    'null_%': (df_raw.isnull().sum() / len(df_raw) * 100)
})

print(info_df)

                      dtype  non_null  null_count  null_%
InvoiceNo            object    541909           0    0.00
StockCode            object    541909           0    0.00
Description          object    540455        1454    0.27
Quantity              int64    541909           0    0.00
InvoiceDate  datetime64[ns]    541909           0    0.00
UnitPrice           float64    541909           0    0.00
CustomerID          float64    406829      135080   24.93
Country              object    541909           0    0.00


In [90]:
df_raw.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.00,541909,541909.00,406829.00
mean,9.55,2011-07-04 13:34:57.156386048,4.61,15287.69
min,-80995.00,2010-12-01 08:26:00,-11062.06,12346.00
25%,1.00,2011-03-28 11:34:00,1.25,13953.00
50%,3.00,2011-07-19 17:17:00,2.08,15152.00
75%,10.00,2011-10-19 11:27:00,4.13,16791.00
max,80995.00,2011-12-09 12:50:00,38970.00,18287.00
std,218.08,NaN,96.76,1713.60


In [91]:
df_raw[df_raw['UnitPrice'] < 0].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [92]:
df_raw[df_raw['Quantity'] < 0].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.00,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.00,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.00,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.00,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.00,United Kingdom


### Cleaning

In [93]:
df = df_raw.copy()
df.shape

(541909, 8)

In [94]:
dp = df.duplicated().sum()
print(f'Duplicates: {dp:,} ({dp/len(df) * 100:.2f}%)')

df = df.drop_duplicates()
print(f'After drop: {df.shape[0]}')

Duplicates: 5,268 (0.97%)
After drop: 536641


In [95]:
before = len(df)
df = df.dropna(subset=['CustomerID'])
after = len(df)
print(f'Removed rows without CustomerID: {before - after:,}')

df['Description'] = df['Description'].fillna('Unknown')

print('\nMissing values after cleanup:')
print(df.isnull().sum())


Removed rows without CustomerID: 135,037

Missing values after cleanup:
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


### Anomalies and Returns

In [96]:
print(f'Quantity rows <= 0:  {(df['Quantity'] <= 0).sum():,}' )
print(f'UnitPrice <= 0:      {(df['UnitPrice'] <= 0).sum():,}')
print(f'Canceled orders (C): {df["InvoiceNo"].astype(str).str.startswith("C").sum():,}')

df_returns = df[df['Quantity'] < 0].copy()
df_sales = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()

Quantity rows <= 0:  8,872
UnitPrice <= 0:      40
Canceled orders (C): 8,872


### Feature Engineering

In [97]:
df_sales = df_sales.copy()

df_sales['Revenue'] = df_sales['Quantity'] * df_sales['UnitPrice']

df_sales['Date'] = pd.to_datetime(df_sales['InvoiceDate']).dt.date
df_sales['InvoiceDate'] = pd.to_datetime(df_sales['InvoiceDate'])

df_sales['Year'] = df_sales['InvoiceDate'].dt.year
df_sales['Month']   = df_sales['InvoiceDate'].dt.month
df_sales['DayOfWeek'] = df_sales['InvoiceDate'].dt.dayofweek
df_sales['Hour']    = df_sales['InvoiceDate'].dt.hour
df_sales['Quarter'] = df_sales['InvoiceDate'].dt.quarter

df_sales['CustomerID'] = df_sales['CustomerID'].astype(int)

print('New columns:')
print(df_sales[['Revenue', 'Year', 'Month', 'DayOfWeek', 'Hour', 'Quarter']].head())


New columns:
   Revenue  Year  Month  DayOfWeek  Hour  Quarter
0    15.30  2010     12          2     8        4
1    20.34  2010     12          2     8        4
2    22.00  2010     12          2     8        4
3    20.34  2010     12          2     8        4
4    20.34  2010     12          2     8        4


### EDA Info

In [98]:
print('=' * 60)
print('Summary')
print('=' * 60)
print(f'Total rows:           {len(df_raw):>10,}')
print(f'After cleaning:       {len(df_sales):>10,}')
print(f'Removed (returns+noise): {len(df_raw) - len(df_sales):>6,}')
print(f'\nUnique customers:    {df_sales["CustomerID"].nunique():>10,}')
print(f'Unique products:     {df_sales["StockCode"].nunique():>10,}')
print(f'Unique countries:    {df_sales["Country"].nunique():>10,}')
print(f'Data period:         {df_sales["InvoiceDate"].min().date()} — {df_sales["InvoiceDate"].max().date()}')
print(f'\nTotal revenue:       £{df_sales["Revenue"].sum():>12,.2f}')
print(f'Average order value:  £{df_sales.groupby("InvoiceNo")["Revenue"].sum().mean():>12,.2f}')

Summary
Total rows:              541,909
After cleaning:          392,692
Removed (returns+noise): 149,217

Unique customers:         4,338
Unique products:          3,665
Unique countries:            37
Data period:         2010-12-01 — 2011-12-09

Total revenue:       £8,887,208.89
Average order value:  £      479.56


In [99]:
df_sales.to_csv('clean_sales.csv', index=False)